In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# CELL 1: Git Clone & Setup
import os
REPO_URL = "https://github.com/DanielQH07/tranSTR_Casual.git" 
REPO_NAME = "tranSTR_Casual"
BRANCH = "full_mode" 

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} -b {BRANCH}
else:
    print("Repo already exists.")

# Change Directory to the repo root 
if os.path.basename(os.getcwd()) != REPO_NAME:
    try:
        target_dir = os.path.join(os.getcwd(), REPO_NAME, "causalvid")
        if os.path.exists(target_dir):
             os.chdir(target_dir)
        elif os.path.exists(REPO_NAME):
             os.chdir(REPO_NAME)
        
        print(f"Changed directory to: {os.getcwd()}")
    except Exception as e:
             print(f"Could not set working directory: {e}")

In [ ]:
# CELL 2: Install & Login W&B (với API Key trực tiếp)
print('=== CELL 2: W&B Setup ===')
!pip install -q wandb --upgrade

import wandb

# ============================================
# WANDB CONFIG - THAY THẾ BẰNG API KEY CỦA BẠN
# ============================================
WANDB_API_KEY = ''
WANDB_PROJECT = 'transtr-causalvid-gdino-nolora-ncod-hum'
WANDB_ENTITY = None

wandb.login(key=WANDB_API_KEY)
print('✅ W&B logged in successfully!')

In [ ]:
# CELL 3: Imports
print('=== CELL 3: Imports ===')
import os, torch, numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from utils.util import set_seed, set_gpu_devices
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import torch.nn.functional as F
print('Imports OK')

In [ ]:
# CELL 4: NCOD + HUM + Verifier/Knowledge training functions
print('=== CELL 4: Functions (NCOD + HUM + Verifier/Knowledge) ===')

def _unpack_batch(batch):
    if len(batch) == 7:
        ff, of, q, a, ans_id, qns_key, q_family_id = batch
    elif len(batch) == 6:
        ff, of, q, a, ans_id, qns_key = batch
        q_family_id = None
    else:
        raise ValueError(f'Unexpected batch format with {len(batch)} elements')
    return ff, of, q, a, ans_id, qns_key, q_family_id

def _compute_sample_indices(qns_keys, key_to_idx, device):
    idxs = [key_to_idx.get(str(k), -1) for k in qns_keys]
    if any(i < 0 for i in idxs):
        missing = [str(qns_keys[i]) for i, v in enumerate(idxs) if v < 0][:5]
        raise KeyError(f'Missing qns_key in key_to_idx mapping: {missing}')
    return torch.tensor(idxs, dtype=torch.long, device=device)

def train_epoch_integrated(
    model, optimizer_model, optimizer_u, U, loader, xe, bce, device, epoch, key_to_idx,
    accumulation_steps=4, hum_alpha=1.0, lambda_verifier=0.2, lambda_knowledge=0.1
):
    model.train()
    total_loss, total_l1, total_l2 = 0.0, 0.0, 0.0
    total_verifier, total_knowledge = 0.0, 0.0
    correct, total = 0, 0
    optimizer_model.zero_grad()
    optimizer_u.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)

        if q_family_id is None:
            q_family_id = torch.zeros_like(tgt)
        else:
            q_family_id = q_family_id.to(device)

        sample_indices = _compute_sample_indices(qns_keys, key_to_idx, device)
        out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
        logits = out['logits']
        fused_logits = out.get('fused_score', logits)
        verifier_logits = out.get('verifier_logits', logits)
        knowledge_logits = out.get('knowledge_score', None)

        probs = torch.softmax(logits, dim=1)
        y_onehot = F.one_hot(tgt, num_classes=logits.size(-1)).float()
        u_batch = U[sample_indices].unsqueeze(1)

        ce_per_sample = -torch.sum(y_onehot * torch.log(torch.clamp(probs, min=1e-12)), dim=1)
        shifted_probs = torch.clamp(probs + (u_batch.detach() * y_onehot), min=1e-12, max=1.0)
        lum_loss = -torch.sum(y_onehot * torch.log(shifted_probs), dim=1)
        hum_loss = (1.0 + hum_alpha * u_batch.detach().squeeze(1)) * ce_per_sample

        hard_mask = (q_family_id >= 2)
        l1 = torch.where(hard_mask, hum_loss, lum_loss).mean()

        verifier_loss = bce(verifier_logits, y_onehot)
        if knowledge_logits is not None:
            knowledge_loss = bce(knowledge_logits, y_onehot)
        else:
            knowledge_loss = torch.tensor(0.0, device=device)

        model_loss = l1 + lambda_verifier * verifier_loss + lambda_knowledge * knowledge_loss
        (model_loss / accumulation_steps).backward()

        shifted_det = probs.detach() + (u_batch * y_onehot)
        l2 = F.mse_loss(shifted_det, y_onehot)
        (l2 / accumulation_steps).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_model.step()
            optimizer_model.zero_grad()
            optimizer_u.step()
            optimizer_u.zero_grad()
            with torch.no_grad():
                U.clamp_(0.0, 0.99)

        total_l1 += l1.item()
        total_l2 += l2.item()
        total_verifier += verifier_loss.item()
        total_knowledge += knowledge_loss.item()
        total_loss += (model_loss + l2).item()
        correct += (fused_logits.argmax(-1) == tgt).sum().item()
        total += tgt.size(0)

        pbar.set_postfix({
            'loss': total_loss / (batch_idx + 1),
            'l1': total_l1 / (batch_idx + 1),
            'l2': total_l2 / (batch_idx + 1),
            'ver': total_verifier / (batch_idx + 1),
            'know': total_knowledge / (batch_idx + 1),
            'acc': correct / max(total, 1) * 100
        })

    n = len(loader)
    return (
        total_loss / n,
        total_l1 / n,
        total_l2 / n,
        total_verifier / n,
        total_knowledge / n,
        correct / max(total, 1) * 100
    )

def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, _, q_family_id = _unpack_batch(batch)
            ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            correct += (logits.argmax(-1) == tgt).sum().item()
            total += tgt.size(0)
    return correct / max(total, 1) * 100

print('Functions defined for integrated training.')

In [ ]:
# CELL 5: Setup Paths & Config
print('=== CELL 5: Paths & Config ===')

# ============================================
# KAGGLE INPUT PATHS - UPDATE THESE!
# ============================================
clip_merged_path = globals().get('CLIP_MERGED_PATH', None)
if clip_merged_path:
    CLIP_FEATURE_PATH = clip_merged_path
else:
    CLIP_FEATURE_PATH = '/kaggle/working/dinov3_T16_dim1024_merge'
GDINO_FEATURE_PATH = '/kaggle/input/datasets/danielq07/gdino-roi-all-nodes-merged'
ANNOTATION_PATH = '/kaggle/input/text-annotation/QA'
SPLIT_DIR = '/kaggle/input/casual-vid-data-split/split'

# Working directories
BASE = '/kaggle/working'
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# Verify paths
print('\n--- Path Verification ---')
def verify_path(name, path):
    if os.path.exists(path):
        items = os.listdir(path)[:3]
        print(f'OK {name}: {items}')
        return True
    print(f'NOT FOUND {name}: {path}')
    return False

all_ok = True
all_ok &= verify_path('DINOv3 Features (1024D)', CLIP_FEATURE_PATH)
all_ok &= verify_path('GroundingDINO ROI Features', GDINO_FEATURE_PATH)
all_ok &= verify_path('Annotations', ANNOTATION_PATH)
all_ok &= verify_path('Splits', SPLIT_DIR)

if not all_ok:
    print('\nPlease update paths above before training.')

# ============================================
# CONFIG - GroundingDINO ROI + Gradient Accumulation + Full DeBERTa tuning
# ============================================
RUN_TRAINING = True
MODEL_FILENAME = 'best_model_gdino_nolora_ncod_hum.ckpt'

FEAT_DIM = 1024
print(f'\nBackbone: DINOv3 (D={FEAT_DIM}) + GroundingDINO ROI')

class Config:
    # Paths
    video_feature_root = CLIP_FEATURE_PATH
    grounding_dino_path = GDINO_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR

    # Model architecture
    topK_frame = 16
    objs = 20
    frames = 16
    select_frames = 5
    topK_obj = 12

    frame_feat_dim = FEAT_DIM
    obj_feat_dim = 1028
    use_grounding_dino = True

    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3

    # Text encoder (no LoRA, full fine-tuning)
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr = 1e-5
    text_pool_mode = 1
    use_lora = False

    # Training
    bs = 8
    accumulation_steps = 4
    lr = 1e-5
    epoch = 5
    gpu = 0
    patience = 5
    gamma = 0.1
    decay = 1e-4
    n_query = 5
    lambda_verifier = 0.3
    lambda_knowledge = 0.2
    return_family_id = True

    # NCOD + HUM
    ncod_u_lr = 0.1
    hum_alpha = 1.0

    # Other
    hard_eval = False
    pos_ratio = 1.0
    neg_ratio = 1.0
    a = 1.0
    num_workers = 4

args = Config()

set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
print(f'Config: frame_feat_dim={args.frame_feat_dim}, obj_feat_dim={args.obj_feat_dim}')
print(f'Using GroundingDINO ROI features: {args.use_grounding_dino}')
print(f'Gradient Accumulation: physical_bs={args.bs} x {args.accumulation_steps} = effective_bs={args.bs * args.accumulation_steps}')
print(f'Verifier loss weight lambda: {args.lambda_verifier}')
print(f'Knowledge loss weight lambda: {args.lambda_knowledge}')
print(f'Return question family id: {args.return_family_id}')
print(f'Full text-encoder fine-tuning: {not args.freeze_text_encoder}')
print(f'LoRA enabled: {args.use_lora}')

In [ ]:
# CELL 6: Create Datasets
print('=== CELL 6: Datasets ===')

train_ds = VideoQADataset(
    split='train', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)

train_loader = DataLoader(train_ds, args.bs, shuffle=True, num_workers=args.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)

train_sample_keys = [f"{row.video_id}_{row.type}" for row in train_ds.sample_list.itertuples(index=False)]
train_key_to_idx = {k: i for i, k in enumerate(train_sample_keys)}

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# CELL 7: Model + Optimizers + NCOD U (No LoRA, full DeBERTa tuning)
print('=== CELL 7: Model ===')
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames
model = VideoQAmodel(**cfg)
model.to(device)

non_text_params = []
text_base_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if 'text_encoder' in name:
        text_base_params.append(p)
    else:
        non_text_params.append(p)

param_groups = []
if len(non_text_params) > 0:
    param_groups.append({'params': non_text_params, 'lr': args.lr, 'weight_decay': args.decay})
if len(text_base_params) > 0:
    param_groups.append({'params': text_base_params, 'lr': args.text_encoder_lr, 'weight_decay': args.decay})
if len(param_groups) == 0:
    raise RuntimeError('No trainable parameters found for optimizer_model.')

optimizer_model = torch.optim.AdamW(param_groups)
scheduler = ReduceLROnPlateau(optimizer_model, 'max', factor=args.gamma, patience=args.patience)

U = torch.nn.Parameter(torch.full((len(train_ds),), 1e-8, dtype=torch.float32, device=device))
optimizer_u = torch.optim.SGD([U], lr=args.ncod_u_lr)

xe = nn.CrossEntropyLoss()
bce = nn.BCEWithLogitsLoss()

BEST_ARTIFACT_NAME = 'best-model-gdino-nolora-ncod-hum'
LAST_ARTIFACT_NAME = 'last-checkpoint-gdino-nolora-ncod-hum'
save_path = os.path.join(MODEL_DIR, MODEL_FILENAME)

print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M')
print(f'Text-encoder trainable params: {sum(p.numel() for p in text_base_params)/1e6:.3f}M')
print(f'U shape: {tuple(U.shape)}')

In [ ]:
# CELL 8: Init W&B + Resume Checkpoint
print('=== CELL 8: Initialize W&B Run ===')

start_epoch = 1
best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
history_rows = []

LATEST_CKPT_PATH = os.path.join(MODEL_DIR, 'latest_checkpoint_gdino_nolora_ncod_hum.ckpt')
TRAIN_HISTORY_CSV_PATH = os.path.join(MODEL_DIR, 'train_history_gdino_nolora_ncod_hum.csv')

RESUME_FROM_CHECKPOINT = False
RESUME_SOURCE = 'local'  # 'local' or 'wandb'
RESUME_ARTIFACT_ALIAS = 'latest'
LOCAL_RESUME_PATH = LATEST_CKPT_PATH

wandb_config = {
    'backbone': 'dinov3+groundingdino',
    'text_encoder': args.text_encoder_type,
    'use_lora': args.use_lora,
    'full_text_finetune': not args.freeze_text_encoder,
    'effective_batch_size': args.bs * args.accumulation_steps,
    'lambda_verifier': args.lambda_verifier,
    'lambda_knowledge': args.lambda_knowledge,
    'ncod_u_lr': args.ncod_u_lr,
    'hum_alpha': args.hum_alpha,
    'resume_enabled': RESUME_FROM_CHECKPOINT,
    'resume_source': RESUME_SOURCE
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, config=wandb_config, reinit=True)
wandb.watch(model, log='gradients', log_freq=100)
print(f'W&B run: {run.url}')

def _load_resume_checkpoint(path, map_location):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Checkpoint not found: {path}')
    return torch.load(path, map_location=map_location)

if RESUME_FROM_CHECKPOINT:
    print(f'Resume enabled from: {RESUME_SOURCE}')
    try:
        checkpoint = None
        resume_path = None

        if RESUME_SOURCE == 'local':
            resume_path = LOCAL_RESUME_PATH
            checkpoint = _load_resume_checkpoint(resume_path, device)

        elif RESUME_SOURCE == 'wandb':
            api = wandb.Api()
            resume_entity = WANDB_ENTITY or api.default_entity
            artifact_path = f'{resume_entity}/{WANDB_PROJECT}/{LAST_ARTIFACT_NAME}:{RESUME_ARTIFACT_ALIAS}'
            print(f'Downloading artifact: {artifact_path}')
            artifact = api.artifact(artifact_path)
            artifact_dir = artifact.download()
            candidate_path = os.path.join(artifact_dir, 'latest_checkpoint_gdino_nolora_ncod_hum.ckpt')
            if os.path.exists(candidate_path):
                resume_path = candidate_path
            else:
                ckpt_files = [f for f in os.listdir(artifact_dir) if f.endswith('.ckpt')]
                if not ckpt_files:
                    raise FileNotFoundError(f'No .ckpt found in artifact folder: {artifact_dir}')
                resume_path = os.path.join(artifact_dir, ckpt_files[0])
            checkpoint = _load_resume_checkpoint(resume_path, device)

        else:
            raise ValueError("RESUME_SOURCE must be 'local' or 'wandb'")

        missing, unexpected = model.load_state_dict(checkpoint['model_state_dict'], strict=False)
        if missing:
            print(f'Warning: missing model keys when resume: {len(missing)}')
        if unexpected:
            print(f'Warning: unexpected model keys when resume: {len(unexpected)}')

        if 'optimizer_model_state_dict' in checkpoint:
            optimizer_model.load_state_dict(checkpoint['optimizer_model_state_dict'])
        if 'optimizer_u_state_dict' in checkpoint:
            optimizer_u.load_state_dict(checkpoint['optimizer_u_state_dict'])
        if 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

        if 'U' in checkpoint:
            with torch.no_grad():
                u_ckpt = checkpoint['U'].to(device).float().view(-1)
                n = min(u_ckpt.numel(), U.numel())
                U[:n].copy_(u_ckpt[:n])
                if u_ckpt.numel() != U.numel():
                    print(f'Warning: U size mismatch (ckpt={u_ckpt.numel()}, current={U.numel()}); copied first {n}')

        best_acc = float(checkpoint.get('best_acc', 0.0))
        start_epoch = int(checkpoint.get('epoch', 0)) + 1
        history = checkpoint.get('history', history)
        history_rows = checkpoint.get('history_rows', history_rows)

        if os.path.exists(TRAIN_HISTORY_CSV_PATH):
            try:
                history_rows = pd.read_csv(TRAIN_HISTORY_CSV_PATH).to_dict('records')
                print(f'Loaded history CSV with {len(history_rows)} rows')
            except Exception as csv_err:
                print(f'Warning: failed to load history CSV: {csv_err}')

        print(f'Resumed from: {resume_path}')
        print(f'Start epoch: {start_epoch} | Best acc: {best_acc:.2f}%')
        wandb.run.summary['resume_path'] = str(resume_path)
        wandb.run.summary['resume_start_epoch'] = int(start_epoch)
        wandb.run.summary['resume_best_acc'] = float(best_acc)
    except Exception as e:
        print(f'Warning: resume failed, starting from scratch. Error: {e}')
else:
    print('Resume disabled. Training starts from epoch 1.')

In [ ]:
# CELL 9: Integrated Training Loop + Checkpoint/CSV Logging
print('=== CELL 9: Training ===')

if RUN_TRAINING:
    for ep in range(start_epoch, args.epoch + 1):
        print(f'\nEpoch {ep}/{args.epoch}')
        total_loss, l1, l2, verifier_loss, knowledge_loss, train_acc = train_epoch_integrated(
            model=model,
            optimizer_model=optimizer_model,
            optimizer_u=optimizer_u,
            U=U,
            loader=train_loader,
            xe=xe,
            bce=bce,
            device=device,
            epoch=ep,
            key_to_idx=train_key_to_idx,
            accumulation_steps=args.accumulation_steps,
            hum_alpha=args.hum_alpha,
            lambda_verifier=args.lambda_verifier,
            lambda_knowledge=args.lambda_knowledge
        )

        val_acc = eval_epoch(model, val_loader, device)
        scheduler.step(val_acc)

        history['train_loss'].append(total_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        epoch_row = {
            'epoch': ep,
            'train_total_loss': float(total_loss),
            'train_l1': float(l1),
            'train_l2': float(l2),
            'train_verifier_loss': float(verifier_loss),
            'train_knowledge_loss': float(knowledge_loss),
            'train_acc': float(train_acc),
            'val_acc': float(val_acc),
            'u_mean': float(U.detach().mean().item()),
            'u_max': float(U.detach().max().item()),
            'lr_main': float(optimizer_model.param_groups[0]['lr'])
        }
        history_rows.append(epoch_row)
        pd.DataFrame(history_rows).to_csv(TRAIN_HISTORY_CSV_PATH, index=False)

        wandb.log(epoch_row)

        is_new_best = val_acc > best_acc
        if is_new_best:
            best_acc = val_acc
            print(f'New best val_acc={best_acc:.2f}%')

        ckpt = {
            'epoch': ep,
            'model_state_dict': model.state_dict(),
            'optimizer_model_state_dict': optimizer_model.state_dict(),
            'optimizer_u_state_dict': optimizer_u.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_acc': best_acc,
            'history': history,
            'history_rows': history_rows,
            'U': U.detach().cpu(),
            'train_sample_keys': train_sample_keys
        }

        torch.save(ckpt, LATEST_CKPT_PATH)

        last_artifact = wandb.Artifact(
            name=LAST_ARTIFACT_NAME,
            type='model',
            metadata={
                'epoch': ep,
                'best_acc': float(best_acc),
                'val_acc': float(val_acc),
                'train_acc': float(train_acc),
                'train_total_loss': float(total_loss)
            }
        )
        last_artifact.add_file(LATEST_CKPT_PATH, name='latest_checkpoint_gdino_nolora_ncod_hum.ckpt')
        if os.path.exists(TRAIN_HISTORY_CSV_PATH):
            last_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name='train_history_gdino_nolora_ncod_hum.csv')
        wandb.log_artifact(last_artifact, aliases=['latest', f'epoch-{ep}'])

        if is_new_best:
            torch.save(ckpt, save_path)
            best_artifact = wandb.Artifact(
                name=BEST_ARTIFACT_NAME,
                type='model',
                metadata={
                    'epoch': ep,
                    'best_acc': float(best_acc),
                    'val_acc': float(val_acc),
                    'train_acc': float(train_acc)
                }
            )
            best_artifact.add_file(save_path, name='best_model_gdino_nolora_ncod_hum.ckpt')
            if os.path.exists(TRAIN_HISTORY_CSV_PATH):
                best_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name='train_history_gdino_nolora_ncod_hum.csv')
            wandb.log_artifact(best_artifact, aliases=['best', f'epoch-{ep}'])

    print(f'Best Val Accuracy: {best_acc:.2f}%')
else:
    print('Skipping training')

In [ ]:
# CELL 10: Detailed Evaluation + CSV export
print('=== CELL 10: Detailed Evaluation ===')
import seaborn as sns

CSV_OUTPUT_PATH = os.path.join(MODEL_DIR, 'test_predictions_gdino_nolora_ncod_hum.csv')

def _build_eval_meta_map(loader):
    dataset = getattr(loader, 'dataset', None)
    sample_list = getattr(dataset, 'sample_list', None) if dataset is not None else None
    meta_map = {}
    if sample_list is None:
        return meta_map
    for _, row in sample_list.iterrows():
        vid = str(row.get('video_id', ''))
        qtype = str(row.get('type', 'unknown'))
        qns_key = f'{vid}_{qtype}'
        meta_map[qns_key] = {
            'video_id': vid,
            'question_type': qtype,
            'question': str(row.get('question', '')),
            'answers': [str(row.get(f'a{i}', '')) for i in range(5)]
        }
    return meta_map

def evaluate_detailed_v2(model, loader, device, log_to_wandb=True):
    model.eval()
    type_results = {}
    prediction_rows = []
    meta_map = _build_eval_meta_map(loader)

    with torch.no_grad():
        for batch in tqdm(loader):
            ff, of, qns, ans_word, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of = ff.to(device), of.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, qns, ans_word, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            preds = probs.argmax(axis=-1)
            targets = ans_id.numpy()

            for i, (key, pred, target, prob_vec) in enumerate(zip(qns_keys, preds, targets, probs)):
                meta = meta_map.get(str(key), {})
                qtype = str(meta.get('question_type', 'unknown'))
                video_id = str(meta.get('video_id', str(key)))
                question = str(meta.get('question', qns[i]))
                answers = meta.get('answers', ['', '', '', '', ''])
                if len(answers) < 5:
                    answers += [''] * (5 - len(answers))
                answers = answers[:5]

                correct_idx = int(target)
                predicted_idx = int(pred)
                is_correct = int(correct_idx == predicted_idx)

                type_results.setdefault(qtype, []).append({
                    'video_id': video_id,
                    'pred': predicted_idx,
                    'target': correct_idx,
                    'correct': bool(is_correct)
                })

                prediction_rows.append({
                    'video_id': video_id,
                    'question_type': qtype,
                    'question': question,
                    'correct_idx': correct_idx,
                    'predicted_idx': predicted_idx,
                    'is_correct': is_correct,
                    'confidence': float(prob_vec[predicted_idx]),
                    'a0': answers[0],
                    'prob_a0': float(prob_vec[0]),
                    'a1': answers[1],
                    'prob_a1': float(prob_vec[1]),
                    'a2': answers[2],
                    'prob_a2': float(prob_vec[2]),
                    'a3': answers[3],
                    'prob_a3': float(prob_vec[3]),
                    'a4': answers[4],
                    'prob_a4': float(prob_vec[4]),
                    'predicted_answer': answers[predicted_idx],
                    'correct_answer': answers[correct_idx]
                })

    prediction_df = pd.DataFrame(prediction_rows)
    prediction_df.to_csv(CSV_OUTPUT_PATH, index=False)
    print(f'Saved detailed predictions CSV: {CSV_OUTPUT_PATH}')

    metrics = {}
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason')
    ]
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        total = len(rows)
        correct = sum(1 for r in rows if r['correct'])
        metrics[name] = (correct / total * 100) if total > 0 else 0.0

    def _calc_hard_metric(type_ans, type_reason):
        if type_ans not in type_results or type_reason not in type_results:
            return 0.0
        ans_by_vid = {r['video_id']: r['correct'] for r in type_results[type_ans]}
        reason_by_vid = {r['video_id']: r['correct'] for r in type_results[type_reason]}
        common_vids = set(ans_by_vid.keys()) & set(reason_by_vid.keys())
        if len(common_vids) == 0:
            return 0.0
        both_correct = sum(1 for vid in common_vids if ans_by_vid[vid] and reason_by_vid[vid])
        return both_correct / len(common_vids) * 100

    metrics['PAR'] = _calc_hard_metric('predictive', 'predictive_reason')
    metrics['CAR'] = _calc_hard_metric('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (
        metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']
    ) / 4.0

    if log_to_wandb:
        wandb.log({
            'eval/Description': metrics['Description'],
            'eval/Explanation': metrics['Explanation'],
            'eval/Predictive_Answer': metrics['Predictive-Answer'],
            'eval/Predictive_Reason': metrics['Predictive-Reason'],
            'eval/Counterfactual_Answer': metrics['Counterfactual-Answer'],
            'eval/Counterfactual_Reason': metrics['Counterfactual-Reason'],
            'eval/PAR': metrics['PAR'],
            'eval/CAR': metrics['CAR'],
            'eval/Acc_ALL': metrics['Acc_ALL']
        })

    print(f"PAR: {metrics['PAR']:.2f}% | CAR: {metrics['CAR']:.2f}% | Acc_ALL: {metrics['Acc_ALL']:.2f}%")
    return metrics, type_results, CSV_OUTPUT_PATH

metrics, raw_results, predictions_csv = evaluate_detailed_v2(model, test_loader, device, log_to_wandb=True)
print(metrics)

In [ ]:
# CELL 11: Finish W&B
print('=== CELL 11: Finish W&B ===')

METRICS_JSON_PATH = os.path.join(MODEL_DIR, 'final_metrics_gdino_nolora_ncod_hum.json')
with open(METRICS_JSON_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Saved metrics JSON: {METRICS_JSON_PATH}')

UPLOAD_CKPT_ARTIFACTS_AT_FINISH = True
if UPLOAD_CKPT_ARTIFACTS_AT_FINISH:
    if os.path.exists(LATEST_CKPT_PATH):
        latest_ckpt_artifact = wandb.Artifact(
            name=LAST_ARTIFACT_NAME,
            type='model',
            metadata={
                'stage': 'finish',
                'checkpoint_kind': 'latest',
                'text_encoder': args.text_encoder_type,
                'lora': args.use_lora,
                'ncod_hum': True
            }
        )
        latest_ckpt_artifact.add_file(LATEST_CKPT_PATH, name='latest_checkpoint_gdino_nolora_ncod_hum.ckpt')
        if os.path.exists(TRAIN_HISTORY_CSV_PATH):
            latest_ckpt_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name='train_history_gdino_nolora_ncod_hum.csv')
        wandb.log_artifact(latest_ckpt_artifact, aliases=['latest', 'finish'])
        print('Uploaded latest checkpoint artifact to W&B.')
    else:
        print(f'Warning: latest checkpoint not found at {LATEST_CKPT_PATH}')

    if os.path.exists(save_path):
        best_ckpt_artifact = wandb.Artifact(
            name=BEST_ARTIFACT_NAME,
            type='model',
            metadata={
                'stage': 'finish',
                'checkpoint_kind': 'best',
                'text_encoder': args.text_encoder_type,
                'lora': args.use_lora,
                'ncod_hum': True
            }
        )
        best_ckpt_artifact.add_file(save_path, name='best_model_gdino_nolora_ncod_hum.ckpt')
        if os.path.exists(TRAIN_HISTORY_CSV_PATH):
            best_ckpt_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name='train_history_gdino_nolora_ncod_hum.csv')
        wandb.log_artifact(best_ckpt_artifact, aliases=['best', 'finish'])
        print('Uploaded best checkpoint artifact to W&B.')
    else:
        print(f'Warning: best checkpoint not found at {save_path}')

final_artifact = wandb.Artifact(
    name='final-results-gdino-nolora-ncod-hum',
    type='results',
    metadata={
        'backbone': 'dinov3+groundingdino',
        'text_encoder': args.text_encoder_type,
        'lora': args.use_lora,
        'ncod_hum': True
    }
)
if os.path.exists(METRICS_JSON_PATH):
    final_artifact.add_file(METRICS_JSON_PATH, name='final_metrics_gdino_nolora_ncod_hum.json')
if os.path.exists(predictions_csv):
    final_artifact.add_file(predictions_csv, name='test_predictions_gdino_nolora_ncod_hum.csv')
if os.path.exists(TRAIN_HISTORY_CSV_PATH):
    final_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name='train_history_gdino_nolora_ncod_hum.csv')
if os.path.exists(LATEST_CKPT_PATH):
    final_artifact.add_file(LATEST_CKPT_PATH, name='latest_checkpoint_gdino_nolora_ncod_hum.ckpt')
if os.path.exists(save_path):
    final_artifact.add_file(save_path, name='best_model_gdino_nolora_ncod_hum.ckpt')
wandb.log_artifact(final_artifact)
wandb.finish()
print('W&B run finished with metrics, CSVs, and checkpoints artifacts.')